In [1]:
import pandas as pd

# Correct file path
path = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\Ausiesuper - Balanced PHD.csv"

# Try reading the CSV (auto-detect delimiter)
df = pd.read_csv(path, sep=None, engine="python", encoding="cp1252")

# Show first 10 rows
print(df.head(10))


  Option Code Option Name Asset Class Filter Sub-Filter  \
0        ARBA    Balanced        Cash    NaN        NaN   
1        ARBA    Balanced        Cash    NaN        NaN   
2        ARBA    Balanced        Cash    NaN        NaN   
3        ARBA    Balanced        Cash    NaN        NaN   
4        ARBA    Balanced        Cash    NaN        NaN   
5        ARBA    Balanced        Cash    NaN        NaN   
6        ARBA    Balanced        Cash    NaN        NaN   
7        ARBA    Balanced        Cash    NaN        NaN   
8        ARBA    Balanced        Cash    NaN        NaN   
9        ARBA    Balanced        Cash    NaN        NaN   

                                        Name            Name Type  \
0  Australia & New Zealand Banking Group Ltd  Name of Institution   
1                         Bank of America NA  Name of Institution   
2                         Bank of America NA  Name of Institution   
3                            Bank of England  Name of Institution   
4    

In [5]:
import pandas as pd
import re

# ===== Constants you can tweak =====
EFFECTIVE_DATE = "31/12/2024"
FUND_NAME      = "AustralianSuper"   # change if needed
# ===================================

# Input and output
in_path  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\Ausiesuper - Balanced PHD.csv"
out_path = r"D:\LinhDao\Programming\SUPERFUNdProject\AusieSuper_Cleaned.csv"

# Currency lookup (first col = Country+currency e.g. "UNITED STATES ... (US Dollar)", second col = ISO 4217 code)
currency_lookup_path = r"D:\LinhDao\Programming\SUPERFUNdProject\CleanedCurrencyCodes.csv"

# Read input
df = pd.read_csv(in_path, encoding="cp1252")

# === Add extra row BEFORE any transformations ===
extra_row = {
    "Option Name": "Balanced",
    "Asset Class": "Fixed Income Private Debt",
    "Filter": "Internally Managed",
    "Name": "Sub Total",
}
extra_df = pd.DataFrame([extra_row], columns=df.columns)  # other columns = NaN
df = pd.concat([df, extra_df], ignore_index=True)

# 1) Remove rows with Asset Class == "Derivatives"
if "Asset Class" in df.columns:
    df = df[df["Asset Class"].astype(str).str.strip().str.lower() != "derivatives"]

# 2) Weighting (%) → always divide by 100
if "Weighting (%)" in df.columns:
    df["Weighting (%)"] = pd.to_numeric(df["Weighting (%)"], errors="coerce") / 100.0

# 3) Append Classification to Name (safe; no duplicates on re-run)
def add_classification_once(row):
    name = str(row.get("Name", "")).strip()
    cls  = row.get("Classification", "")
    if pd.isna(cls): return name
    cls = str(cls).strip()
    if not cls: return name
    tag = f"@{cls}"
    if re.search(rf"(^|\s){re.escape(tag)}(\s|$)", name):  # already appended
        return name
    return f"{name} {tag}".strip()

if {"Name","Classification"}.issubset(df.columns):
    df["Name"] = df.apply(add_classification_once, axis=1)

# 4) Fill "$ Value" from "Value Range" (ranges averaged; single values taken as-is)
def parse_value_range(value_range):
    """
    - Ranges: "$10m to $50m", "100m-300m"  -> average of endpoints
    - Single value (optional < or >): "> $1.5bn", "< $2m", "$20m" -> that number
    Units: k, m, b/bn
    """
    if pd.isna(value_range): return None
    s = str(value_range).strip().lower().replace(",", "")
    s = re.sub(r"\s+", " ", s)

    unit_scale = {"k": 1e3, "m": 1e6, "b": 1e9, "bn": 1e9, "": 1.0}

    # Range
    m = re.match(r'^\$?(\d+(?:\.\d+)?)(k|m|b|bn)?\s*(?:to|-)\s*\$?(\d+(?:\.\d+)?)(k|m|b|bn)?$', s)
    if m:
        n1, u1, n2, u2 = m.groups()
        u1 = u1 or ""; u2 = u2 or u1
        return ((float(n1)*unit_scale[u1]) + (float(n2)*unit_scale[u2])) / 2.0

    # Single (with optional < or >)
    m = re.match(r'^[<>]?\s*\$?(\d+(?:\.\d+)?)(k|m|b|bn)?$', s)
    if m:
        n, u = m.groups()
        u = u or ""
        return float(n) * unit_scale[u]

    return None

if {"$ Value","Value Range"}.issubset(df.columns):
    is_blank = df["$ Value"].astype(str).str.strip().eq("") | df["$ Value"].isna()
    has_range = df["Value Range"].astype(str).str.strip().ne("")
    to_fill = is_blank & has_range
    df.loc[to_fill, "$ Value"] = df.loc[to_fill, "Value Range"].apply(parse_value_range)

# 5) Int/Ext from Filter / Sub-Filter (Externally Managed -> 1, Internally Managed -> 0)
if "Int/Ext" not in df.columns:
    df["Int/Ext"] = pd.NA  # will cast to Int64 later

def _contains(series, phrase_regex):
    return series.astype(str).str.contains(phrase_regex, case=False, na=False, regex=True)

flt  = df.get("Filter", pd.Series([""]*len(df)))
sflt = df.get("Sub-Filter", pd.Series([""]*len(df)))

ext_mask = _contains(flt, r"externally managed") | _contains(sflt, r"externally managed")
int_mask = _contains(flt, r"internally managed") | _contains(sflt, r"internally managed")

df.loc[ext_mask, "Int/Ext"] = 1
df.loc[int_mask, "Int/Ext"] = 0

# 6) Update Asset Class with Listed/Unlisted prefix and Sub-Filter override
ac = df.get("Asset Class", pd.Series([""]*len(df))).astype(str)
listed_mask   = _contains(flt,  r"\blisted\b")
unlisted_mask = _contains(flt,  r"\bunlisted\b")

def _prefix_if_needed(current: str, prefix: str) -> str:
    cur = current.strip()
    if cur.lower().startswith(prefix.lower() + " ") or cur.lower() == prefix.lower():
        return cur
    return f"{prefix} {cur}".strip()

df.loc[listed_mask,   "Asset Class"] = ac[listed_mask].apply(lambda x: _prefix_if_needed(x, "Listed"))
df.loc[unlisted_mask, "Asset Class"] = ac[unlisted_mask].apply(lambda x: _prefix_if_needed(x, "Unlisted"))

# Override by Sub-Filter exact match
fipd_mask = sflt.astype(str).str.strip().str.casefold().eq("fixed income private debt")
df.loc[fipd_mask, "Asset Class"] = "Fixed Income Private Debt"

# 7) Final cleanup
# 7a) Fill empty Int/Ext with 1 and keep integer dtype
if "Int/Ext" in df.columns:
    df["Int/Ext"] = df["Int/Ext"].fillna(1).astype("Int64")

# 7b) In Name: exact "Total" or exact "nan" (string) or true NaN -> "Sub Total"
if "Name" in df.columns:
    name_orig = df["Name"]
    name_stripped = name_orig.astype(str).str.strip()
    to_sub_total = name_stripped.eq("Total") | name_stripped.eq("nan") | name_orig.isna()
    df.loc[to_sub_total, "Name"] = "Sub Total"

# 8) Drop unwanted columns (after all logic)
drop_cols = [
    "Option Code", "Filter", "Sub-Filter", "Name Type", "Issuer Type",
    "Actual Currency Exposure (%)", "Actual Asset Allocation (%)",
    "Effect of Derivatives Exposure (%)", "Classification", "Sort Order",
    "Value Range", "Geo Latitude", "Geo Longitude"
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# 9) Rename columns to final schema names
rename_map = {
    "Asset Class": "Asset Class Name",
    "Name": "Name/Kind of Investment Item",
    "Security Identifier": "Stock ID",
    "Location": "Listed Country",
    "$ Value": "Value (AUD)",
    "Weighting (%)": "Weighting",
}
df = df.rename(columns=rename_map)

# === Fuzzy Currency lookup: replace with matched ISO code only ===
try:
    lu_cur = pd.read_csv(currency_lookup_path, encoding="cp1252", usecols=[0, 1])
    lu_cur.columns = ["Country", "Code"]

    # Tokenizer: include both country and currency words; drop punctuation/noise; light stopwords
    STOP = {"the", "of", "and"}
    def _tokens(s: str):
        if pd.isna(s): return set()
        s = str(s).lower()
        s = re.sub(r"[^a-z0-9]+", " ", s)
        toks = [t for t in s.split() if t and t not in STOP]
        return set(toks)

    # Precompute lookup tokens
    lookup_pairs = [(_tokens(c), str(code).strip().upper()) for c, code in zip(lu_cur["Country"], lu_cur["Code"])]

    # Aliases for common variations
    ALIAS = {
        "usa": "united states",
        "us": "united states",
        "uk": "united kingdom",
        "uae": "united arab emirates",
        "south korea": "korea republic of",
        "north korea": "korea democratic people s republic of",
        "russia": "russian federation",
        "viet nam": "vietnam",
    }

    def _normalize_text(s: str):
        s1 = str(s or "")
        for k, v in ALIAS.items():
            s1 = re.sub(rf"\b{re.escape(k)}\b", v, s1, flags=re.IGNORECASE)
        return s1

    def _best_code(original_text):
        if pd.isna(original_text) or str(original_text).strip() == "":
            return ""
        txt = _normalize_text(original_text)
        t = _tokens(txt)
        if not t: 
            return ""

        best_score, best = 0.0, None
        for toks, code in lookup_pairs:
            if not toks: continue
            inter = len(t & toks)
            if inter == 0: continue
            union = len(t | toks)
            score = inter / union if union else 0.0
            if score > best_score:
                best_score, best = score, code

        return best if best_score > 0 else ""

    if "Currency" in df.columns:
        df["Currency"] = df["Currency"].apply(_best_code)

except Exception as e:
    print(f"⚠️ Currency lookup skipped due to error: {e}")


# 10) Add Effective Date and Fund Name (hardcoded)
df["Effective Date"] = EFFECTIVE_DATE
df["Fund Name"] = FUND_NAME

# 11) Reorder columns to final order
final_order = [
    "Effective Date", "Fund Name", "Option Name", "Asset Class Name", "Int/Ext",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country",
    "Units Held", "% Ownership", "Address", "Value (AUD)", "Weighting"
]
# Make sure every requested column exists (create empty if missing)
for col in final_order:
    if col not in df.columns:
        df[col] = ""

df = df[final_order]

# Save cleaned output
df.to_csv(out_path, index=False, encoding="cp1252")
print(f"✅ Cleaned file saved to: {out_path}")


✅ Cleaned file saved to: D:\LinhDao\Programming\SUPERFUNdProject\AusieSuper_Cleaned.csv


C:\Users\thuon\AppData\Local\Temp\ipykernel_37852\4242874037.py:124: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Int/Ext"] = df["Int/Ext"].fillna(1).astype("Int64")


In [ ]:
# new code aug 20
import pandas as pd
import re

# ===== Constants you can tweak =====
FUND_NAME = "AustralianSuper"   # change if needed
# ===================================

# Input and output
in_path  = r"D:\LinhDao\Programming\SUPERFUNdProject\Linh-working-files\Ausiesuper - Balanced PHD.csv"
out_path = r"D:\LinhDao\Programming\SUPERFUNdProject\AusieSuper_Cleaned_final.csv"

# Currency lookup
currency_lookup_path = r"D:\LinhDao\Programming\SUPERFUNdProject\CleanedCurrencyCodes.csv"

# Read input
df = pd.read_csv(in_path, encoding="cp1252")

# === Add extra row BEFORE any transformations ===
extra_row = {
    "Option Name": "Balanced",
    "Asset Class": "Fixed Income Private Debt",
    "Filter": "Internally Managed",
    "Name": "Sub Total",
}
extra_df = pd.DataFrame([extra_row], columns=df.columns)
df = pd.concat([df, extra_df], ignore_index=True)

# 1) Remove rows with Asset Class == "Derivatives"
if "Asset Class" in df.columns:
    df = df[df["Asset Class"].astype(str).str.strip().str.lower() != "derivatives"]

# 2) Weighting (%) → divide by 100
if "Weighting (%)" in df.columns:
    df["Weighting (%)"] = pd.to_numeric(df["Weighting (%)"], errors="coerce") / 100.0

# 3) Append Classification to Name
def add_classification_once(row):
    name = str(row.get("Name", "")).strip()
    cls  = row.get("Classification", "")
    if pd.isna(cls): return name
    cls = str(cls).strip()
    if not cls: return name
    tag = f"@{cls}"
    if re.search(rf"(^|\s){re.escape(tag)}(\s|$)", name):  
        return name
    return f"{name} {tag}".strip()

if {"Name","Classification"}.issubset(df.columns):
    df["Name"] = df.apply(add_classification_once, axis=1)

# 4) Fill "$ Value" from "Value Range"
def parse_value_range(value_range):
    if pd.isna(value_range): return None
    s = str(value_range).strip().lower().replace(",", "")
    s = re.sub(r"\s+", " ", s)
    unit_scale = {"k": 1e3, "m": 1e6, "b": 1e9, "bn": 1e9, "": 1.0}

    m = re.match(r'^\$?(\d+(?:\.\d+)?)(k|m|b|bn)?\s*(?:to|-)\s*\$?(\d+(?:\.\d+)?)(k|m|b|bn)?$', s)
    if m:
        n1, u1, n2, u2 = m.groups()
        u1 = u1 or ""; u2 = u2 or u1
        return ((float(n1)*unit_scale[u1]) + (float(n2)*unit_scale[u2])) / 2.0

    m = re.match(r'^[<>]?\s*\$?(\d+(?:\.\d+)?)(k|m|b|bn)?$', s)
    if m:
        n, u = m.groups(); u = u or ""
        return float(n) * unit_scale[u]
    return None

if {"$ Value","Value Range"}.issubset(df.columns):
    is_blank = df["$ Value"].astype(str).str.strip().eq("") | df["$ Value"].isna()
    has_range = df["Value Range"].astype(str).str.strip().ne("")
    to_fill = is_blank & has_range
    df.loc[to_fill, "$ Value"] = df.loc[to_fill, "Value Range"].apply(parse_value_range)

# 5) Int/Ext from Filter / Sub-Filter
if "Int/Ext" not in df.columns:
    df["Int/Ext"] = pd.NA

def _contains(series, phrase_regex):
    return series.astype(str).str.contains(phrase_regex, case=False, na=False, regex=True)

flt  = df.get("Filter", pd.Series([""]*len(df)))
sflt = df.get("Sub-Filter", pd.Series([""]*len(df)))

ext_mask = _contains(flt, r"externally managed") | _contains(sflt, r"externally managed")
int_mask = _contains(flt, r"internally managed") | _contains(sflt, r"internally managed")

df.loc[ext_mask, "Int/Ext"] = 1
df.loc[int_mask, "Int/Ext"] = 0

# 6) Update Asset Class with Listed/Unlisted prefix and Sub-Filter override
ac = df.get("Asset Class", pd.Series([""]*len(df))).astype(str)
listed_mask   = _contains(flt,  r"\blisted\b")
unlisted_mask = _contains(flt,  r"\bunlisted\b")

def _prefix_if_needed(current: str, prefix: str) -> str:
    cur = current.strip()
    if cur.lower().startswith(prefix.lower() + " ") or cur.lower() == prefix.lower():
        return cur
    return f"{prefix} {cur}".strip()

df.loc[listed_mask,   "Asset Class"] = ac[listed_mask].apply(lambda x: _prefix_if_needed(x, "Listed"))
df.loc[unlisted_mask, "Asset Class"] = ac[unlisted_mask].apply(lambda x: _prefix_if_needed(x, "Unlisted"))

fipd_mask = sflt.astype(str).str.strip().str.casefold().eq("fixed income private debt")
df.loc[fipd_mask, "Asset Class"] = "Fixed Income Private Debt"

# 7) Final cleanup
df["Int/Ext"] = (
    pd.Series(df["Int/Ext"])
    .replace("", pd.NA)
    .astype("Int64")
    .fillna(1)
    .astype(int)
)

if "Name" in df.columns:
    name_orig = df["Name"]
    name_stripped = name_orig.astype(str).str.strip()
    to_sub_total = name_stripped.eq("Total") | name_stripped.eq("nan") | name_orig.isna()
    df.loc[to_sub_total, "Name"] = "Sub Total"

# 8) Drop unwanted columns
drop_cols = [
    "Option Code", "Filter", "Sub-Filter", "Name Type", "Issuer Type",
    "Actual Currency Exposure (%)", "Actual Asset Allocation (%)",
    "Effect of Derivatives Exposure (%)", "Classification", "Sort Order",
    "Value Range", "Geo Latitude", "Geo Longitude"
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# 9) Rename columns
rename_map = {
    "Asset Class": "Asset Class Name",
    "Name": "Name/Kind of Investment Item",
    "Security Identifier": "Stock ID",
    "Location": "Listed Country",
    "$ Value": "Value (AUD)",
    "Weighting (%)": "Weighting",
}
df = df.rename(columns=rename_map)

# === NEW: Normalize Fixed Income Private Debt ===
if "Asset Class Name" in df.columns:
    mask_fipd = df["Asset Class Name"].astype(str).str.strip().str.casefold().eq("fixed income private debt")
    df.loc[mask_fipd, "Asset Class Name"] = "Fixed Income (Private Debt)"

# === Currency lookup stays unchanged ===
try:
    lu_cur = pd.read_csv(currency_lookup_path, encoding="cp1252", usecols=[0, 1])
    lu_cur.columns = ["Country", "Code"]

    STOP = {"the", "of", "and"}
    def _tokens(s: str):
        if pd.isna(s): return set()
        s = str(s).lower()
        s = re.sub(r"[^a-z0-9]+", " ", s)
        return {t for t in s.split() if t and t not in STOP}

    lookup_pairs = [(_tokens(c), str(code).strip().upper()) for c, code in zip(lu_cur["Country"], lu_cur["Code"])]

    ALIAS = {
        "usa": "united states",
        "us": "united states",
        "uk": "united kingdom",
        "uae": "united arab emirates",
        "south korea": "korea republic of",
        "north korea": "korea democratic people s republic of",
        "russia": "russian federation",
        "viet nam": "vietnam",
    }

    def _normalize_text(s: str):
        s1 = str(s or "")
        for k, v in ALIAS.items():
            s1 = re.sub(rf"\b{re.escape(k)}\b", v, s1, flags=re.IGNORECASE)
        return s1

    def _best_code(original_text):
        if pd.isna(original_text) or str(original_text).strip() == "":
            return ""
        txt = _normalize_text(original_text)
        t = _tokens(txt)
        if not t: return ""
        best_score, best = 0.0, None
        for toks, code in lookup_pairs:
            if not toks: continue
            inter = len(t & toks)
            if inter == 0: continue
            union = len(t | toks)
            score = inter / union if union else 0.0
            if score > best_score:
                best_score, best = score, code
        return best if best_score > 0 else ""

    if "Currency" in df.columns:
        df["Currency"] = df["Currency"].apply(_best_code)

except Exception as e:
    print(f"⚠️ Currency lookup skipped due to error: {e}")

# 10) Add Effective Date + Fund Name
df["Effective Date"] = pd.Timestamp(2024, 12, 31)
df["Fund Name"] = FUND_NAME

# 11) Coerce numerics
for col in ["Units Held", "Value (AUD)"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# 12) Reorder
final_order = [
    "Effective Date", "Fund Name", "Option Name", "Asset Class Name", "Int/Ext",
    "Name/Kind of Investment Item", "Currency", "Stock ID", "Listed Country",
    "Units Held", "% Ownership", "Address", "Value (AUD)", "Weighting"
]
for col in final_order:
    if col not in df.columns:
        df[col] = ""
df = df[final_order]

# 13) Save
df.to_csv(out_path, index=False, encoding="cp1252", date_format="%b %d %Y")
print(f"✅ Cleaned file saved to: {out_path}")



✅ Cleaned file saved to: D:\LinhDao\Programming\SUPERFUNdProject\AusieSuper_Cleaned_final.csv
